1. Environment Setup

This notebook prepares the environment for our Mini-BART research project.

In [ ]:
# %pip install -q "numpy==1.26.4" "transformers<5" datasets evaluate rouge_score sentencepiece accelerate

In [ ]:
# %pip install -q "scipy==1.15.3"

In [2]:
%pip install -q "numpy==2.2.6" "transformers==4.57.6" "datasets==3.6.0" evaluate rouge_score sentencepiece accelerate tensorboard pandas matplotlib pyyaml

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


2. Imports

Import the libraries required for this project.

In [ ]:
import numpy as np
import scipy
import torch
import transformers

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

NumPy: 2.4.6
SciPy: 1.16.3
PyTorch: 2.14.1+cpu
Transformers: 5.22.1
CUDA available: False


3. GPU Verification

Verify that PyTorch can access our Colab GPU.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cpu
CUDA available: False


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

PyTorch: 2.14.1+cpu
CUDA available: False
GPU: No GPU detected


In [1]:
import torch
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA: True
GPU: Tesla T4


In [1]:
import torch
import numpy as np
import scipy
import transformers
import datasets
import evaluate
import huggingface_hub
import fsspec

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

print("\nNumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("fsspec:", fsspec.__version__)
print("\nAll imports successful!")

PyTorch: 2.14.1+cu130
CUDA: True
GPU: Tesla T4

NumPy: 2.2.6
SciPy: 1.16.3
Transformers: 4.57.6
Datasets: 3.6.0
HuggingFace Hub: 0.36.2
fsspec: 2025.3.0

All imports successful!


4. Configuration

Define project-wide settings.

## 4. Loading the Pretrained BART Teacher

We use facebook/bart-large-cnn as our teacher model.

It is already pretrained and fine-tuned for CNN/DailyMail summarization.

Our objective is to use this teacher to guide the training of a smaller BART student through knowledge distillation.

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

teacher = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

teacher = teacher.to("cuda")
teacher.eval()

print("Teacher loaded successfully!")
print("Model:", MODEL_NAME)
print("Parameters:", sum(p.numel() for p in teacher.parameters()))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Teacher loaded successfully!
Model: facebook/bart-large-cnn
Parameters: 406290432


In [3]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount("/content/drive")

# Create a permanent storage location
TEACHER_DIR = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"
os.makedirs(TEACHER_DIR, exist_ok=True)

# Save the model and tokenizer
teacher.save_pretrained(TEACHER_DIR)
tokenizer.save_pretrained(TEACHER_DIR)

print("Teacher model and tokenizer saved successfully!")
print("Location:", TEACHER_DIR)


Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


Teacher model and tokenizer saved successfully!
Location: /content/drive/MyDrive/research_artifacts/mini_bart/teacher


In [4]:
print(os.listdir(TEACHER_DIR))

['config.json', 'generation_config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.json', 'merges.txt', 'tokenizer.json']


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

teacher_path = "/content/drive/MyDrive/research_artifacts/mini_bart/teacher"

test_tokenizer = AutoTokenizer.from_pretrained(teacher_path)
test_model = AutoModelForSeq2SeqLM.from_pretrained(teacher_path)

print("Reload successful!")
print("Parameters:", sum(p.numel() for p in test_model.parameters()))

/usr/local/lib/python3.13/dist-packages/transformers/models/bart/configuration_bart.py:177: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


Reload successful!
Parameters: 406290432
